In [ ]:
from google.colab import files
import pandas as pd

print("RAW bank.csv upload karo (pipeline isse shuru hoga):")
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
df_raw = pd.read_csv(INPUT_FILE, sep=None, engine='python')
print(f"Raw data ready: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")


RAW bank.csv upload karo (pipeline isse shuru hoga):


Saving bank.csv to bank.csv
Raw data ready: 11162 rows, 17 columns


In [ ]:

%%writefile config.yaml
# Pipeline configuration — settings yahan se control hoti hain
input_file: "bank.csv"
output_dir: "results"

profiling:
  outlier_threshold_pct: 5.0

cleaning:
  sentinel_columns:
    pdays: -1
  cap_columns: ["balance", "duration", "campaign", "pdays", "previous"]

validation:
  isolation_forest_contamination: 0.02
  rules:
    age: [18, 100]
    day: [1, 31]

Writing config.yaml


In [ ]:
%%writefile pipeline.py
"""
Automated Data Quality & Validation Pipeline (Module 4)
CadetX Virtual Work Experience

Usage: python pipeline.py --input bank.csv --config config.yaml
Stages: Profile -> Clean -> Validate
"""
import argparse
import json
import logging
import os
from datetime import datetime

import numpy as np
import pandas as pd
import yaml
from sklearn.ensemble import IsolationForest

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.FileHandler('pipeline.log'),
                              logging.StreamHandler()])
log = logging.getLogger('dq-pipeline')


def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)


def profile(df, out_dir, threshold_pct):
    log.info('MODULE 1: Profiling started')
    suspicious = []
    for col in df.columns:
        issues = []
        s = df[col]
        if float(s.isna().mean() * 100) > 50:
            issues.append('very high missingness')
        if pd.api.types.is_numeric_dtype(s):
            vals = s.dropna()
            q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
            iqr = q3 - q1
            n_out = int(((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)).sum())
            if n_out > len(df) * threshold_pct / 100:
                issues.append(f'heavy outliers ({n_out} rows)')
        if issues:
            suspicious.append({'column': col, 'issues': issues})
    report = {'dataset': {'rows': int(len(df)), 'columns': int(df.shape[1])},
              'generated_at': datetime.now().isoformat(timespec='seconds'),
              'suspicious_columns': suspicious}
    with open(os.path.join(out_dir, 'profiling_report.json'), 'w') as f:
        json.dump(report, f, indent=2)
    log.info('MODULE 1: done — %d suspicious columns', len(suspicious))


def clean(df, cfg):
    log.info('MODULE 2: Cleaning started')
    cleaning_log = []
    df_clean = df.copy()

    for col, sentinel in cfg['cleaning']['sentinel_columns'].items():
        if col in df_clean.columns:
            mask = df_clean[col] == sentinel
            n = int(mask.sum())
            df_clean.loc[mask, col] = np.nan
            cleaning_log.append({'action': 'sentinel_to_nan', 'column': col, 'rows_affected': n})
            log.info('  sentinel fix: %s (%d rows)', col, n)

    if 'balance' in df_clean.columns:
        mask = df_clean['balance'] < 0
        n = int(mask.sum())
        df_clean.loc[mask, 'balance'] = np.nan
        cleaning_log.append({'action': 'invalid_to_nan', 'column': 'balance', 'rows_affected': n})

    n_cap = 0
    for col in cfg['cleaning']['cap_columns']:
        if col not in df_clean.columns:
            continue
        s = df_clean[col]
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask = (s < lo) | (s > hi)
        n_cap += int(mask.sum())
        df_clean[col] = s.clip(lo, hi)
    cleaning_log.append({'action': 'outlier_capping', 'rows_affected': n_cap})

    df_clean.to_csv(os.path.join(cfg['output_dir'], 'cleaned_data.csv'), index=False)
    with open(os.path.join(cfg['output_dir'], 'cleaning_log.json'), 'w') as f:
        json.dump({'actions': cleaning_log}, f, indent=2)
    log.info('MODULE 2: done — %d outlier cells capped', n_cap)
    return df_clean


def validate(df_clean, cfg, out_dir):
    log.info('MODULE 3: Validation started')
    violations = 0
    for col, (lo, hi) in cfg['validation']['rules'].items():
        if col in df_clean.columns:
            n = int(((df_clean[col] < lo) | (df_clean[col] > hi)).sum())
            violations += n
            log.info('  rule %s in [%s,%s]: %d violations', col, lo, hi, n)

    feats = [c for c in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
             if c in df_clean.columns]
    X = df_clean[feats].fillna(df_clean[feats].median())
    iso = IsolationForest(contamination=cfg['validation']['isolation_forest_contamination'],
                          random_state=42)
    anomalies = int((iso.fit_predict(X) == -1).sum())

    health = round(100 - (violations / len(df_clean) * 100) - (anomalies / len(df_clean) * 100), 2)
    with open(os.path.join(out_dir, 'validation_report.json'), 'w') as f:
        json.dump({'rule_violations': violations, 'anomalies': anomalies,
                   'dataset_health_score': health}, f, indent=2)
    log.info('MODULE 3: done — health %.2f/100', health)
    return health


def main():
    ap = argparse.ArgumentParser(description='Data Quality Pipeline')
    ap.add_argument('--input', required=True)
    ap.add_argument('--config', default='config.yaml')
    args = ap.parse_args()

    cfg = load_config(args.config)
    os.makedirs(cfg['output_dir'], exist_ok=True)

    log.info('PIPELINE START: %s', args.input)
    df = pd.read_csv(args.input, sep=None, engine='python')
    log.info('Loaded %d rows x %d columns', len(df), df.shape[1])

    profile(df, cfg['output_dir'], cfg['profiling']['outlier_threshold_pct'])
    df_clean = clean(df, cfg)
    health = validate(df_clean, cfg, cfg['output_dir'])

    log.info('PIPELINE COMPLETE — Health Score: %.2f/100', health)
    print(f'Pipeline complete! Health Score: {health}/100 -> outputs in results/')


if __name__ == '__main__':
    main()

Writing pipeline.py


In [ ]:
!python pipeline.py --input bank.csv --config config.yaml

2026-10-06 20:36:24,317 [INFO] PIPELINE START: bank.csv
2026-10-06 20:36:24,448 [INFO] Loaded 11162 rows x 17 columns
2026-10-06 20:36:24,448 [INFO] MODULE 1: Profiling started
2026-10-06 20:36:24,480 [INFO] MODULE 1: done — 5 suspicious columns
2026-10-06 20:36:24,480 [INFO] MODULE 2: Cleaning started
2026-10-06 20:36:24,482 [INFO]   sentinel fix: pdays (8324 rows)
2026-10-06 20:36:24,583 [INFO] MODULE 2: done — 3504 outlier cells capped
2026-10-06 20:36:24,583 [INFO] MODULE 3: Validation started
2026-10-06 20:36:24,584 [INFO]   rule age in [18,100]: 0 violations
2026-10-06 20:36:24,585 [INFO]   rule day in [1,31]: 0 violations
2026-10-06 20:36:24,970 [INFO] MODULE 3: done — health 97.99/100
2026-10-06 20:36:24,971 [INFO] PIPELINE COMPLETE — Health Score: 97.99/100
Pipeline complete! Health Score: 97.99/100 -> outputs in results/


In [ ]:
%%writefile test_pipeline.py
import os
import pandas as pd
import yaml
import pipeline

cfg = yaml.safe_load(open('config.yaml'))
cfg['output_dir'] = 'test_results'

df = pd.read_csv('bank.csv', sep=None, engine='python')
n_raw = len(df)

def test_load():
    assert df.shape[1] == 17

def test_profile_outputs():
    pipeline.profile(df, 'test_results', 5.0)
    assert os.path.exists('test_results/profiling_report.json')

def test_clean_sentinel_fixed():
    df_clean = pipeline.clean(df, cfg)
    assert (df_clean['pdays'] == -1).sum() == 0

def test_clean_no_negative_balance():
    df_clean = pipeline.clean(df, cfg)
    assert (df_clean['balance'] < 0).sum() == 0

def test_rows_preserved():
    df_clean = pipeline.clean(df, cfg)
    assert len(df_clean) == n_raw

def test_health_above_90():
    df_clean = pipeline.clean(df, cfg)
    health = pipeline.validate(df_clean, cfg, 'test_results')
    assert health > 90

Writing test_pipeline.py


In [ ]:
!pytest -q

.FFFFF                                                                   [100%]
=================================== FAILURES ===================================
_____________________________ test_profile_outputs _____________________________

    def test_profile_outputs():
>       pipeline.profile(df, 'test_results', 5.0)

test_pipeline.py:16: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

df =        age          job  marital  education  ... pdays  previous poutcome deposit
0       59       admin.  married  se...re      no
11161   34   technician  married  secondary  ...    -1         0  unknown      no

[11162 rows x 17 columns]
out_dir = 'test_results', threshold_pct = 5.0

    def profile(df, out_dir, threshold_pct):
        log.info('MODULE 1: Profiling started')
        suspicious = []
        for col in df.columns:
            issues = []
            s = df[col]
            if float(s.isna().mean() * 100) > 50:
                issues.append('

In [ ]:
src = open('pipeline.py').read()
src = src.replace(
    "def profile(df, out_dir, threshold_pct):\n    log.info",
    "def profile(df, out_dir, threshold_pct):\n    os.makedirs(out_dir, exist_ok=True)\n    log.info")
src = src.replace(
    "def clean(df, cfg):\n    log.info",
    "def clean(df, cfg):\n    os.makedirs(cfg['output_dir'], exist_ok=True)\n    log.info")
open('pipeline.py', 'w').write(src)
print('pipeline.py patched — ab har function khud folder banayega')

pipeline.py patched — ab har function khud folder banayega


In [ ]:
!pytest -q

......                                                                   [100%]
6 passed in 2.42s


In [ ]:
!python pipeline.py --input bank.csv --config config.yaml

2026-10-06 20:47:36,479 [INFO] PIPELINE START: bank.csv
2026-10-06 20:47:36,611 [INFO] Loaded 11162 rows x 17 columns
2026-10-06 20:47:36,611 [INFO] MODULE 1: Profiling started
2026-10-06 20:47:36,633 [INFO] MODULE 1: done — 5 suspicious columns
2026-10-06 20:47:36,634 [INFO] MODULE 2: Cleaning started
2026-10-06 20:47:36,636 [INFO]   sentinel fix: pdays (8324 rows)
2026-10-06 20:47:36,727 [INFO] MODULE 2: done — 3504 outlier cells capped
2026-10-06 20:47:36,728 [INFO] MODULE 3: Validation started
2026-10-06 20:47:36,728 [INFO]   rule age in [18,100]: 0 violations
2026-10-06 20:47:36,729 [INFO]   rule day in [1,31]: 0 violations
2026-10-06 20:47:37,119 [INFO] MODULE 3: done — health 97.99/100
2026-10-06 20:47:37,119 [INFO] PIPELINE COMPLETE — Health Score: 97.99/100
Pipeline complete! Health Score: 97.99/100 -> outputs in results/


In [10]:
%%writefile requirements.txt
pandas
numpy
pyyaml
scikit-learn
pytest

Writing requirements.txt


In [11]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY pipeline.py config.yaml ./
COPY bank.csv ./
CMD ["python", "pipeline.py", "--input", "bank.csv", "--config", "config.yaml"]

Writing Dockerfile


In [12]:
files.download('pipeline.py')
files.download('config.yaml')
files.download('test_pipeline.py')
files.download('Dockerfile')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>